# Chapter 0 - Welcome to Sam

It is Monday, week one at Drift Entertainment. Priya, the Head of Data, points to last quarter's investor slide: Jazz was reported as the fastest-growing genre, then walked back when a one-off query used the wrong revenue logic.

Your assignment is simple and high stakes: in two weeks, build Sam, an AI analyst the leadership team can trust in Monday Q&A. Today starts with the foundation: deterministic data and a clean workspace.

## Learn
- Why Priya insists every teammate sees the same numbers
- What Sam's workspace will contain by the end of today
- Who at Drift will ask Sam what, and why that context matters


In [ ]:
-- Build core scaffolding
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS SNOWFLAKE_EXAMPLE
  COMMENT = 'DEMO: Shared walkthrough database (Expires: 2026-04-18)';

CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.GIT_REPOS
  COMMENT = 'DEMO: Shared git repository clones (Expires: 2026-04-18)';

CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.SAM_DRIFT
  COMMENT = 'DEMO: Drift Entertainment walkthrough schema (Expires: 2026-04-18)';

CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS
  COMMENT = 'DEMO: Shared semantic views for walkthroughs (Expires: 2026-04-18)';

CREATE OR REPLACE WAREHOUSE SFE_SAM_SNOWMAN_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'DEMO: Sam walkthrough warehouse (Expires: 2026-04-18)';

GRANT USAGE ON WAREHOUSE SFE_SAM_SNOWMAN_WH TO ROLE SYSADMIN;
GRANT OPERATE ON WAREHOUSE SFE_SAM_SNOWMAN_WH TO ROLE SYSADMIN;

USE ROLE SYSADMIN;
USE WAREHOUSE SFE_SAM_SNOWMAN_WH;

CREATE OR REPLACE GIT REPOSITORY SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO
  API_INTEGRATION = SFE_GITHUB_API_INTEGRATION
  ORIGIN = 'https://github.com/sfc-gh-miwhitaker/Sam-the-Snowman.git'
  COMMENT = 'DEMO: Sam repository clone for walkthrough notebooks (Expires: 2026-04-18)';

ALTER GIT REPOSITORY SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO FETCH;


In [ ]:
-- Create Drift source tables
USE ROLE SYSADMIN;
USE WAREHOUSE SFE_SAM_SNOWMAN_WH;
USE DATABASE SNOWFLAKE_EXAMPLE;
USE SCHEMA SAM_DRIFT;

CREATE OR REPLACE TABLE ARTIST (ARTIST_ID NUMBER, NAME STRING) COMMENT = 'DEMO: Drift artist dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE ALBUM (ALBUM_ID NUMBER, TITLE STRING, ARTIST_ID NUMBER) COMMENT = 'DEMO: Drift album dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE GENRE (GENRE_ID NUMBER, NAME STRING) COMMENT = 'DEMO: Drift genre dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE MEDIA_TYPE (MEDIA_TYPE_ID NUMBER, NAME STRING) COMMENT = 'DEMO: Drift media type dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE TRACK (TRACK_ID NUMBER, NAME STRING, ALBUM_ID NUMBER, MEDIA_TYPE_ID NUMBER, GENRE_ID NUMBER, COMPOSER STRING, MILLISECONDS NUMBER, BYTES NUMBER, UNIT_PRICE NUMBER(10,2)) COMMENT = 'DEMO: Drift track dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE PLAYLIST (PLAYLIST_ID NUMBER, NAME STRING) COMMENT = 'DEMO: Drift playlist dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE PLAYLIST_TRACK (PLAYLIST_ID NUMBER, TRACK_ID NUMBER) COMMENT = 'DEMO: Drift playlist-track bridge (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE EMPLOYEE (EMPLOYEE_ID NUMBER, LAST_NAME STRING, FIRST_NAME STRING, TITLE STRING, REPORTS_TO NUMBER, BIRTH_DATE DATE, HIRE_DATE DATE, ADDRESS STRING, CITY STRING, STATE STRING, COUNTRY STRING, POSTAL_CODE STRING, PHONE STRING, FAX STRING, EMAIL STRING) COMMENT = 'DEMO: Drift employee dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE CUSTOMER (CUSTOMER_ID NUMBER, FIRST_NAME STRING, LAST_NAME STRING, COMPANY STRING, ADDRESS STRING, CITY STRING, STATE STRING, COUNTRY STRING, POSTAL_CODE STRING, PHONE STRING, FAX STRING, EMAIL STRING, SUPPORT_REP_ID NUMBER) COMMENT = 'DEMO: Drift customer dimension (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE INVOICE (INVOICE_ID NUMBER, CUSTOMER_ID NUMBER, INVOICE_DATE DATE, BILLING_ADDRESS STRING, BILLING_CITY STRING, BILLING_STATE STRING, BILLING_COUNTRY STRING, BILLING_POSTAL_CODE STRING, TOTAL NUMBER(10,2)) COMMENT = 'DEMO: Drift invoice fact (Expires: 2026-04-18)';
CREATE OR REPLACE TABLE INVOICE_LINE (INVOICE_LINE_ID NUMBER, INVOICE_ID NUMBER, TRACK_ID NUMBER, UNIT_PRICE NUMBER(10,2), QUANTITY NUMBER) COMMENT = 'DEMO: Drift invoice line fact (Expires: 2026-04-18)';


In [ ]:
-- Load deterministic Parquet files from the repo stage
COPY INTO ARTIST FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/ARTIST.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO ALBUM FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/ALBUM.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO GENRE FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/GENRE.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO MEDIA_TYPE FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/MEDIA_TYPE.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO TRACK FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/TRACK.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO PLAYLIST FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/PLAYLIST.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO PLAYLIST_TRACK FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/PLAYLIST_TRACK.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO EMPLOYEE FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/EMPLOYEE.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO CUSTOMER FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/CUSTOMER.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO INVOICE FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/INVOICE.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO INVOICE_LINE FROM @SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_SAM_THE_SNOWMAN_REPO/branches/main/datasets/drift/INVOICE_LINE.parquet FILE_FORMAT=(TYPE=PARQUET) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;


In [ ]:
-- Check
SHOW TABLES IN SCHEMA SNOWFLAKE_EXAMPLE.SAM_DRIFT;

SELECT 'ARTIST' AS table_name, COUNT(*) AS row_count FROM ARTIST
UNION ALL SELECT 'TRACK', COUNT(*) FROM TRACK
UNION ALL SELECT 'CUSTOMER', COUNT(*) FROM CUSTOMER
UNION ALL SELECT 'INVOICE', COUNT(*) FROM INVOICE
UNION ALL SELECT 'INVOICE_LINE', COUNT(*) FROM INVOICE_LINE
ORDER BY table_name;


## Reflect
Priya closes your setup review with one line: "Good. Same data, same numbers, every time. That is the floor."

## Next
Tomorrow morning before standup, she wants Sam to answer its first business question without hand-holding. Open [`ch01_first_agent.ipynb`](ch01_first_agent.ipynb).
